# Decision matrix cases (PDF Table 1) — `recombination_updated.ipynb`

This notebook loads **`NonDuplicationRecombination`** from `recombination_updated.ipynb` (code cells 1 and 4 only) so the implementation is not duplicated here.

Table 1 in *GRG Recombination* crosses **ancestral interval** \(I_u\) vs query interval \(I_w = [L,R)\) with **node mutations** \(M_u\) vs \(I_w\). Each row below is one minimal GRG and one call to the same `_recurse_attach` used by `recombine` / `recombine_multi`, starting from internal node **u** (root **R** → **u** → sample **0**) so the first visit is exactly the matrix row/column you want.

Sentinel mutations on sample **0** at positions 0 and 99 set `bp_range` to \([0,100)\) without affecting \(I_u\) (ancestors-only span for **u**).

In [ ]:
from pathlib import Path
import nbformat


def _notebook_dir():
    try:
        here = Path(__file__).resolve().parent
        if (here / "recombination_updated.ipynb").exists():
            return here
    except NameError:
        pass
    cwd = Path.cwd().resolve()
    for d in [cwd, *cwd.parents]:
        if (d / "recombination_updated.ipynb").exists():
            return d
        nested = d / "grgl" / "recombination" / "recombination_updated.ipynb"
        if nested.exists():
            return nested.parent
    raise FileNotFoundError(
        "Could not find recombination_updated.ipynb (search cwd and parents for "
        "recombination_updated.ipynb or grgl/recombination/recombination_updated.ipynb)."
    )


_UPDATED = _notebook_dir() / "recombination_updated.ipynb"
_nb = nbformat.read(_UPDATED.open(), as_version=4)
_ns = {}
for _idx in (1, 4):
    exec(compile(_nb.cells[_idx].source, f"{_UPDATED}:{_idx}", "exec"), _ns)

NonDuplicationRecombination = _ns["NonDuplicationRecombination"]
NEGATIVE_NODE_IDS = _ns["NEGATIVE_NODE_IDS"]


In [ ]:
from pygrgl import MutableGRG, Mutation

CASE_LABELS = {
    1: "Complete inheritance (direct edge, stop)",
    2: "All of Mu in Iw; Iu disjoint — split (bubble all Mu), stop",
    3: "All of Mu in Iw; Iu partial overlap — split, recurse",
    4: "Mu disjoint from Iw; Iu subset Iw — recurse (path compression)",
    5: "Mu disjoint from Iw; Iu disjoint — stop (prune)",
    6: "Mu disjoint from Iw; Iu partial overlap — recurse",
    7: "Partial Mu in Iw; Iu subset Iw — split, recurse",
    8: "Partial Mu in Iw; Iu disjoint — split, stop",
    9: "Partial Mu in Iw; Iu partial overlap — split, recurse",
}


def classify_matrix_case(Iu, L, R, has_all_relevant, has_no_relevant, has_partial_relevant):
    """Map (Iu, Mu) vs Iw to case id 1..9 (PDF Table 1). Iu is ancestors-only; None => not in table."""
    if Iu is None:
        return 0, "ROOT (no parent ancestry)"
    iu_disjoint = R <= Iu[0] or L >= Iu[1]
    iu_subset = Iu[0] >= L and Iu[1] <= R
    if iu_disjoint:
        iu_col = 1
    elif iu_subset:
        iu_col = 0
    else:
        iu_col = 2
    if has_no_relevant:
        mu_row = 1
    elif has_all_relevant:
        mu_row = 0
    else:
        mu_row = 2
    case_id = iu_col + mu_row * 3 + 1
    return case_id, CASE_LABELS.get(case_id, "?")


class InstrumentedRecombination(NonDuplicationRecombination):
    """Logs PDF matrix case at each _recurse_attach entry (before mutating visited)."""

    def __init__(self, grg, case_log):
        super().__init__(grg)
        self._case_log = case_log

    def _recurse_attach(self, node_id, offspring_id, L, R, visited=None, connected=None):
        if L >= R:
            return
        if visited is None:
            visited = set()
        if connected is None:
            connected = set()
        if node_id in visited:
            return

        all_muts = self._get_node_mutations(node_id)
        relevant_muts = self._get_mutations_in_interval(node_id, L, R)
        all_mut_ids = {m[0] for m in all_muts}
        relevant_mut_ids = {m[0] for m in relevant_muts}
        has_all_relevant = (relevant_mut_ids == all_mut_ids) and len(all_mut_ids) > 0
        has_no_relevant = len(relevant_mut_ids) == 0
        has_partial_relevant = len(relevant_mut_ids) > 0 and relevant_mut_ids != all_mut_ids
        Iu = self._get_ancestral_coverage(node_id)
        case_id, label = classify_matrix_case(
            Iu, L, R, has_all_relevant, has_no_relevant, has_partial_relevant
        )
        self._case_log.append((case_id, label, node_id, L, R))

        return super()._recurse_attach(node_id, offspring_id, L, R, visited, connected)


In [ ]:
R, U, SAMPLE = 2, 1, 0


def build_base_grg():
    """Root R -> internal u -> sample 0; sentinels on sample for bp_range [0, 100)."""
    g = MutableGRG(1, 1, True)
    g.make_node()
    g.make_node()
    g.connect(R, U)
    g.connect(U, SAMPLE)
    g.add_mutation(Mutation(0, "A", "G"), SAMPLE)
    g.add_mutation(Mutation(99, "C", "T"), SAMPLE)
    return g


def case_1_grg():
    g = build_base_grg()
    g.add_mutation(Mutation(10, "A", "G"), R)
    g.add_mutation(Mutation(15, "C", "T"), U)
    return g


def case_2_grg():
    g = build_base_grg()
    g.add_mutation(Mutation(10, "A", "G"), R)
    g.add_mutation(Mutation(11, "C", "T"), R)
    g.add_mutation(Mutation(50, "A", "G"), U)
    g.add_mutation(Mutation(60, "C", "T"), U)
    return g


def case_3_grg():
    g = build_base_grg()
    g.add_mutation(Mutation(10, "A", "G"), R)
    g.add_mutation(Mutation(49, "C", "T"), R)
    g.add_mutation(Mutation(25, "A", "G"), U)
    g.add_mutation(Mutation(35, "C", "T"), U)
    return g


def case_4_grg():
    g = build_base_grg()
    g.add_mutation(Mutation(15, "A", "G"), R)
    g.add_mutation(Mutation(20, "C", "T"), R)
    g.add_mutation(Mutation(80, "A", "G"), U)
    return g


def case_5_grg():
    g = build_base_grg()
    g.add_mutation(Mutation(70, "A", "G"), R)
    g.add_mutation(Mutation(80, "C", "T"), U)
    return g


def case_6_grg():
    g = build_base_grg()
    g.add_mutation(Mutation(10, "A", "G"), R)
    g.add_mutation(Mutation(40, "C", "T"), R)
    g.add_mutation(Mutation(80, "A", "G"), U)
    return g


def case_7_grg():
    g = build_base_grg()
    g.add_mutation(Mutation(15, "A", "G"), R)
    g.add_mutation(Mutation(25, "C", "T"), R)
    g.add_mutation(Mutation(20, "A", "G"), U)
    g.add_mutation(Mutation(80, "C", "T"), U)
    return g


def case_8_grg():
    g = build_base_grg()
    g.add_mutation(Mutation(70, "A", "G"), R)
    g.add_mutation(Mutation(80, "C", "T"), R)
    g.add_mutation(Mutation(25, "A", "G"), U)
    g.add_mutation(Mutation(90, "C", "T"), U)
    return g


def case_9_grg():
    g = build_base_grg()
    g.add_mutation(Mutation(10, "A", "G"), R)
    g.add_mutation(Mutation(55, "C", "T"), R)
    g.add_mutation(Mutation(25, "A", "G"), U)
    g.add_mutation(Mutation(80, "C", "T"), U)
    return g


MATRIX_SCENARIOS = [
    (1, case_1_grg, U, 0, 30),
    (2, case_2_grg, U, 40, 70),
    (3, case_3_grg, U, 20, 40),
    (4, case_4_grg, U, 0, 50),
    (5, case_5_grg, U, 0, 50),
    (6, case_6_grg, U, 30, 50),
    (7, case_7_grg, U, 0, 50),
    (8, case_8_grg, U, 0, 50),
    (9, case_9_grg, U, 20, 45),
]


In [ ]:
def run_all_matrix_cases():
    results = []
    for expected_id, builder, u, L, R in MATRIX_SCENARIOS:
        NEGATIVE_NODE_IDS.clear()
        g = builder()
        log = []
        recomb = InstrumentedRecombination(g, log)
        offspring_id = g.make_node(negative=True)
        recomb._recurse_attach(u, offspring_id, L, R, visited=set(), connected=set())
        row = next((e for e in log if e[2] == u), None)
        assert row is not None, (expected_id, log)
        got_id, label, _, _, _ = row
        assert got_id == expected_id, (
            f"Case {expected_id}: classified as {got_id} ({label}) for u={u} Iw=[{L},{R})"
        )
        results.append((expected_id, label, u, L, R))
    return results


_out = run_all_matrix_cases()
for cid, lab, u, L, R in _out:
    print(f"Case {cid} OK — u={u}, Iw=[{L},{R}) — {lab}")
print(f"\nAll {len(_out)} PDF Table 1 cells exercised on node u={U}.")
